# 05 — Privacy Concerns (Edoardo Balzano)

This notebook focuses on the privacy implications of the student early-warning system. It explores:
1. **Sensitive Feature Handling**: Identifying and masking demographic features.
2. **k-Anonymity**: Assessing re-identification risk in the OULAD dataset.
3. **Privacy-Preserving Explanations**: Ensuring SHAP/LIME explanations do not leak sensitive personal traits.

In [ ]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(str(Path.cwd().parent))

from src.privacy import (
    apply_feature_masking, 
    check_k_anonymity, 
    suppress_sensitive_from_explanation,
    SENSITIVE_COLS
)

data_dir = Path.cwd().parent / "data"

# Load shared artifacts
try:
    model   = joblib.load(data_dir / "base_model.pkl")
    X_train = joblib.load(data_dir / "X_train.pkl")
    X_test  = joblib.load(data_dir / "X_test.pkl")
    y_test  = joblib.load(data_dir / "y_test.pkl")
    print("Successfully loaded shared artifacts.")
except FileNotFoundError:
    print("Error: Shared artifacts not found in data/. Please run 02_base_model.ipynb first.")

## 1. k-Anonymity Assessment & Enforcement

We evaluate if the dataset provides enough anonymity for students based on quasi-identifiers. If not, we apply generalization and suppression.

In [ ]:
from src.privacy import enforce_k_anonymity, generalize_categorical, get_default_generalization_maps

# Define quasi-identifiers
quasi_ids = ["gender", "region", "highest_education", "age_band"]

# Step A: Initial check (via enforcement with 0 suppression to see counts)
print("Checking initial k-anonymity...")
df_anon = enforce_k_anonymity(X_test, quasi_ids, k=5, action='suppress')

# Step B: Apply Generalization to reduce violations
maps = get_default_generalization_maps()
X_test_gen = X_test.copy()
for col, mapping in maps.items():
    X_test_gen = generalize_categorical(X_test_gen, col, mapping)

print("\nChecking k-anonymity after generalization...")
df_anon_gen = enforce_k_anonymity(X_test_gen, quasi_ids, k=5, action='suppress')

print(f"\nData retained after suppression: {len(df_anon_gen) / len(X_test) * 100:.2f}%")

## 3. Privacy-Preserving Explanations (Advanced XAI Guard)

Explanations (like SHAP values) can leak sensitive information even if the sensitive features themselves are removed, because other features act as 'proxies'. We detect these proxies and suppress them from the output.

In [ ]:
from src.privacy import detect_proxy_features, suppress_sensitive_from_explanation

# Step 1: Detect Proxy Features
# We check which features are highly correlated with sensitive traits in X_test
proxies = detect_proxy_features(X_test, SENSITIVE_COLS, threshold=0.3)

# Step 2: Simulate an explanation (using dummy SHAP values for demonstration)
dummy_shap = pd.Series(
    data=[0.5, 0.2, 0.1, 0.8, -0.1, 0.4],
    index=["weighted_score", "total_clicks", "gender_enc", "region_enc", "imd_band_enc", "submission_rate"]
)

print("\nOriginal Explanation (Feature Importances):")
print(dummy_shap)

secure_shap = suppress_sensitive_from_explanation(dummy_shap, SENSITIVE_COLS, proxy_cols=proxies)

print("\nPrivacy-Preserving Explanation:")
print(secure_shap)